1__Load Dataset

In [6]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

df = pd.read_csv("raw_wholesale_customers.csv")
df.head()
     

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


2__Select Features + IQR Cap

In [7]:
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

X = df[FEATURES].copy()

def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in FEATURES:
    low, high = iqr_fun(X[col])
    X[col] = X[col].clip(lower=low, upper=high)

df[FEATURES] = X
X.describe().round(2)

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.00,440.00,440.00,440.00,440.00,440.00
mean,11357.57,5048.59,7236.38,2507.09,2392.62,1266.72
std,10211.54,4386.38,6596.53,2408.30,2940.79,1083.07
min,3.00,55.00,3.00,25.00,3.00,3.00
25%,3127.75,1533.00,2153.00,742.25,256.75,408.25
50%,8504.00,3627.00,4755.50,1526.00,816.50,965.50
75%,16933.75,7190.25,10655.75,3554.25,3922.00,1820.25
max,37642.75,15676.12,23409.88,7772.25,9419.88,3938.25


3__Scale Features

In [8]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled shape:", X_scaled.shape)

Scaled shape: (440, 6)


4__Elbow Method

In [9]:

sse = {}
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    sse[k] = km.inertia_

for k, val in sse.items():
    print(f"k={k}  SSE={val:.2f}")
     

k=1  SSE=2640.00
k=2  SSE=1728.19
k=3  SSE=1363.45
k=4  SSE=1202.41
k=5  SSE=1070.15
k=6  SSE=964.76
k=7  SSE=921.14
k=8  SSE=776.63
k=9  SSE=726.88
k=10  SSE=707.41


5__Train K-Means

In [10]:
kmeans = KMeans(n_clusters=5, n_init="auto", random_state=42)
km_labels = kmeans.fit_predict(X_scaled)

df["KMeans_Cluster"] = km_labels.astype(int)
df["KMeans_Cluster"].value_counts().sort_index()
     

KMeans_Cluster
0     76
1    191
2     25
3     88
4     60
Name: count, dtype: int64

6__Evaluate K-Means

In [11]:
sil_km = silhouette_score(X_scaled, km_labels)
dbi_km = davies_bouldin_score(X_scaled, km_labels)

print(f"Silhouette Score : {sil_km:.3f}  (closer to +1 is better)")
print(f"Davies-Bouldin   : {dbi_km:.3f}  (lower is better)")
     

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"
centers_df.round(2)

Silhouette Score : 0.283  (closer to +1 is better)
Davies-Bouldin   : 1.270  (lower is better)


,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
Cluster,,,,,,
0,9202.67,6833.30,9104.12,1326.16,3280.12,1871.76
1,8376.23,2150.65,3160.63,1646.33,779.25,674.02
2,17461.54,13805.60,17524.12,4120.57,5460.56,3583.64
3,22346.70,3409.14,3969.33,5819.60,583.07,1566.95
4,4916.98,10768.85,18350.13,1212.37,7780.02,981.37


7__Second Algorithm: Agglomerative Clustering

In [12]:
agg = AgglomerativeClustering(n_clusters=5, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)

df["Agg_Cluster"] = agg_labels.astype(int)
df["Agg_Cluster"].value_counts().sort_index()

Agg_Cluster
0     70
1     72
2    164
3     55
4     79
Name: count, dtype: int64

8__Compare Methods

In [13]:

sil_agg = silhouette_score(X_scaled, agg_labels)
dbi_agg = davies_bouldin_score(X_scaled, agg_labels)

print(f"K-Means       — Silhouette: {sil_km:.3f}  Davies-Bouldin: {dbi_km:.3f}")
print(f"Agglomerative — Silhouette: {sil_agg:.3f}  Davies-Bouldin: {dbi_agg:.3f}")

K-Means       — Silhouette: 0.283  Davies-Bouldin: 1.270
Agglomerative — Silhouette: 0.218  Davies-Bouldin: 1.325


9__Sanity Check (3 Clients)

In [14]:
sample_idx = [0, 1, 2]
cols = ["Channel", "Region"] + FEATURES + ["KMeans_Cluster", "Agg_Cluster"]
df.loc[sample_idx, cols]

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,KMeans_Cluster,Agg_Cluster
0,2,3,12669.0,9656.0,7561.0,214.0,2674.0,1338.00,0,4
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0,4
2,2,3,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25,0,0


10__Save Output

In [15]:
df.to_csv("segmented_wholesale_customers.csv", index=False)
print("Saved to dataset/segmented_wholesale_customers.csv")

OSError: Cannot save file into a non-existent directory: 'dataset'